# Actor-Critic — A2C (Advantage Actor-Critic), PPO (Proximal Policy Optimization)

_Modern Deep Learning & AI_

**One net picks actions, a second net judges the state. Update the actor by how much better an action did than expected.**

Plain policy gradients use the full episode return $G$, which is noisy. Actor-Critic reduces that noise.

     It uses two networks. The actor is the policy: it picks actions. The critic estimates the value $V(s)$: how good a state is.

---

This notebook is a practice scaffold for the **Actor-Critic — A2C (Advantage Actor-Critic), PPO (Proximal Policy Optimization)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch
import torch.nn as nn
from torch.distributions import Categorical

class ActorCritic(nn.Module):
    def __init__(self, state_dim, n_actions):
        super().__init__()
        self.body   = nn.Sequential(nn.Linear(state_dim, 32), nn.ReLU())
        self.actor  = nn.Linear(32, n_actions)   # policy logits
        self.critic = nn.Linear(32, 1)           # value V(s)
    def forward(self, s):
        h = self.body(s)
        return self.actor(h), self.critic(h).squeeze(-1)

ac = ActorCritic(state_dim=4, n_actions=2)
opt = torch.optim.Adam(ac.parameters(), lr=1e-3)
gamma = 0.9

# synthetic one-step batch (s, a, r, s')
s, s2 = torch.randn(6, 4), torch.randn(6, 4)
r = torch.randn(6)

logits, V = ac(s)
with torch.no_grad():
    _, V2 = ac(s2)
    target = r + gamma * V2                      # TD target for critic
    adv = target - V                             # advantage A(s, a)

dist = Categorical(logits=logits)
a = dist.sample()
actor_loss  = -(dist.log_prob(a) * adv).mean()   # scale by advantage
critic_loss = nn.functional.mse_loss(V, target)
loss = actor_loss + 0.5 * critic_loss
opt.zero_grad(); loss.backward(); opt.step()
print(float(actor_loss), float(critic_loss))

## Visualize the data & results

_On the same corridor, does actor-critic learn, and does the critic baseline make it smoother?_

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(2)
N, GOAL = 8, 7                       # same corridor, goal reward = +10
theta = np.zeros((N, 2))            # actor: softmax logits per state
V = np.zeros(N)                    # critic: state value baseline
lr_a, lr_c, gamma = 0.4, 0.3, 0.95
rewards = []
for ep in range(300):
    s, total = 0, 0.0
    for _ in range(30):
        p = np.exp(theta[s] - theta[s].max()); p /= p.sum()
        a = rng.choice(2, p=p)
        s2 = min(N - 1, s + 1) if a == 1 else max(0, s - 1)
        r = 10.0 if s2 == GOAL else -0.1
        adv = r + gamma * V[s2] - V[s]          # one-step TD advantage
        V[s] += lr_c * adv                       # critic update
        grad = -p; grad[a] += 1.0
        theta[s] += lr_a * adv * grad            # actor scaled by advantage
        s, total = s2, total + r
        if s2 == GOAL:
            break
    rewards.append(total)

sm = np.convolve(rewards, np.ones(20) / 20, mode='valid')
plt.figure(figsize=(6, 4))
plt.plot(sm, color='#c89bff', label='episode reward')
plt.xlabel('episode'); plt.ylabel('smoothed episode reward')
plt.title('Actor-Critic on 8-state corridor'); plt.legend()
plt.tight_layout(); plt.show()